In [ ]:
import numpy as np
import torch
import hockey.hockey_env as h_env

from memory import ReplayBuffer
from sac import SACAgent

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
env = h_env.HockeyEnv()
player2 = h_env.BasicOpponent(weak=False)

ac_space = env.action_space
o_space = env.observation_space
print(ac_space)
print(o_space)
print(list(zip(env.observation_space.low, env.observation_space.high)))

In [ ]:
max_episodes=int(1e5)
max_steps=500

buffer = ReplayBuffer()
agent = SACAgent(env.observation_space.shape[0], env.action_space.shape[0], noise_seq_len=int(1e5), device = device)

In [ ]:
ob,_info = env.reset()
print(ob)
agent.actor(torch.FloatTensor(ob).unsqueeze(0).to(device))

In [ ]:
stats = []
losses = []

In [ ]:
for i in range(max_episodes):
    # print("Starting a new episode")    
    total_reward = []
    ob, _info = env.reset(True)
    obs_agent2 = env.obs_agent_two()
    done = False
    for t in range(max_steps):
        with torch.no_grad():
            a1, _ = agent.actor.sample(torch.FloatTensor(ob).unsqueeze(0).to(device))
        a1 = a1.cpu().numpy()[0]
        a2 = player2.act(obs_agent2)

        (ob_new, reward, done, trunc, _info) = env.step(np.hstack([a1,a2]))
        buffer.add((ob, a1, reward, ob_new, float(done)))
        total_reward.append(reward)
        ob=ob_new        
        obs_agent2 = env.obs_agent_two()
        if done: 
            break
    agent.update(buffer)
    stats.append([i,total_reward,t+1])
    if (i+1)%20 == 0:
        agent.actor.gen.reset()
    
    if ((i-1)%20==0):
        print("{}: Reward: {}".format(i, np.sum(total_reward)))

In [ ]:
o, info = env.reset()
_ = env.render()
player2 = h_env.BasicOpponent(weak=False)

In [ ]:
obs_buffer = []
reward_buffer=[]
obs, info = env.reset()
obs_agent2 = env.obs_agent_two()
for _ in range(251):
    env.render()
    with torch.no_grad():
        a1, _ = agent.actor.sample(torch.FloatTensor(ob).unsqueeze(0).to(device))
        a1 = a1.cpu().numpy()[0]
    a2 = player2.act(obs_agent2)

    obs, r, d, t, info = env.step(np.hstack([a1,a2]))    
    obs_buffer.append(obs)
    reward_buffer.append(r)
    obs_agent2 = env.obs_agent_two()
    if d or t: 
        break
obs_buffer = np.asarray(obs_buffer)
reward_buffer = np.asarray(reward_buffer)

In [ ]:
env.close()